<a href="https://colab.research.google.com/github/tipismaca/Trabajo-Bebidas-del-Maipo-SA/blob/main/Codigo/Entrega2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caso de Estudio: Bebidas del Maipo S.A. — Entrega 2
**Autoras:** Maytte Almuna, Macarena Campos, Darlyn Jerez y Valentina Roa


## 1. Configuración inicial
Clonación del repositorio y librerías

In [66]:
# Clona el repositorio para tener acceso a los datos
!rm -rf Trabajo-Bebidas-del-Maipo-SA
!git clone https://github.com/tipismaca/Trabajo-Bebidas-del-Maipo-SA.git

Cloning into 'Trabajo-Bebidas-del-Maipo-SA'...
remote: Enumerating objects: 579, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 579 (delta 85), reused 35 (delta 35), pack-reused 470 (from 1)
Receiving objects: 100% (579/579), 35.68 MiB | 13.06 MiB/s, done.
Resolving deltas: 100% (308/308), done.


In [67]:
%pip install -q "pulp>=2.7,<3"   # PuLP 2.x trae CBC incluido
!pip install -q highspy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pulp
from IPython.display import display


## 2. Carga de datos

In [68]:
ruta_base = "Trabajo-Bebidas-del-Maipo-SA/Datos_crudos/"
ruta_resultados = "Trabajo-Bebidas-del-Maipo-SA/Resultados/"

---
## Parte A1. Balance de inventario por sku y semana.


In [73]:
# =====================================================================
# 2.1 DATOS CRUDOS DEL HORIZONTE TÁCTICO
# =====================================================================
maestro      = pd.read_csv(ruta_base + "maestro_sku.csv")
inv_inicial  = pd.read_csv(ruta_base + "inventario_inicial.csv")
capacidad    = pd.read_csv(ruta_base + "capacidad_horizonte.csv")
setups       = pd.read_csv(ruta_base + "setups_cip.csv")
compromisos  = pd.read_csv(ruta_base + "compromisos_semana_focal.csv")


# =====================================================================
# 2.2 TABLAS DE TRASPASO DE LA ENTREGA 1 (Parte C)
# =====================================================================
tabla_c1 = pd.read_csv(ruta_resultados + "merma_por_linea.csv")
tabla_c2 = pd.read_csv(ruta_resultados + "sobrellenado_por_formato.csv")
tabla_c3 = pd.read_csv(ruta_resultados + "pronostico_sku_horizonte.csv")
tabla_c4 = pd.read_csv(ruta_resultados + "sigma_error_sku_horizonte.csv")


# C.1: ρ_ℓ por línea
rho = dict(zip(tabla_c1["Línea"], tabla_c1["ρ (proporción)"].astype(float)))


# C.2: litros de jarabe adicionales por caja, por formato (se usa en A3)
jarabe_extra_L_caja = dict(zip(tabla_c2["Formato (L)"].astype(float),
                               tabla_c2["Jarabe adicional (L/caja)"].astype(float)))


# C.3 y C.4: matriz SKU × (h=1..h=13) a formato largo (sku, h, valor)
def matriz_a_largo(tabla, nombre_valor):
    largo = tabla.melt(id_vars=["Familia", "SKU"], var_name="h", value_name=nombre_valor)
    largo["h"] = largo["h"].str.replace("h=", "", regex=False).astype(int)
    largo[nombre_valor] = largo[nombre_valor].astype(float)
    return largo.rename(columns={"SKU": "sku", "Familia": "familia"})


pronostico = matriz_a_largo(tabla_c3, "d_hat")
sigma_err  = matriz_a_largo(tabla_c4, "sigma")


# =====================================================================
# 2.3 VALIDACIONES DE CONSISTENCIA ENTRE E1 Y LOS DATOS DE E2
# =====================================================================
SKUS   = maestro["sku"].tolist()
LINEAS = sorted(maestro["linea_envasado"].unique())
T      = sorted(capacidad["t"].astype(int).tolist())


assert set(rho) == set(LINEAS), f"C.1 no trae ρ para todas las líneas: {rho}"
assert all(0 <= r < 1 for r in rho.values()), "ρ_ℓ debe estar en [0, 1)."
assert set(jarabe_extra_L_caja) == set(maestro["formato_L"]), "C.2 no cubre todos los formatos."
for nombre, df in [("C.3", pronostico), ("C.4", sigma_err)]:
    assert set(df["sku"]) == set(SKUS), f"{nombre}: los SKU no coinciden con maestro_sku."
    assert set(df["h"]) == set(T), f"{nombre}: los horizontes no son {T[0]}..{T[-1]}."
    assert not df.duplicated(["sku", "h"]).any(), f"{nombre}: hay (sku, h) duplicados."
assert (pronostico["d_hat"] >= 0).all(), "Hay pronósticos negativos en C.3."
assert (sigma_err["sigma"] >= 0).all(), "Hay σ̂ negativos en C.4."
assert set(inv_inicial["sku"]) == set(SKUS), "inventario_inicial no cubre los 9 SKU."


# El horizonte h del pronóstico corresponde a la semana t = h del plan
# (h = 1 es el lunes 28/09/2026, primera fila de capacidad_horizonte.csv).
d_hat = {(r.sku, r.h): r.d_hat for r in pronostico.itertuples()}


# Aviso: pronósticos que superan 3 veces la mediana del SKU en el horizonte
mediana_sku = pronostico.groupby("sku")["d_hat"].transform("median")
picos = pronostico[pronostico["d_hat"] > 3 * mediana_sku].merge(
    capacidad[["t", "fecha_lunes"]], left_on="h", right_on="t")
if len(picos):
    picos["veces_mediana"] = (picos["d_hat"]
                              / picos["sku"].map(pronostico.groupby("sku")["d_hat"].median())).round(1)
    print("⚠ Pronósticos atípicos (> 3 × mediana del SKU):")
    display(picos[["sku", "h", "fecha_lunes", "d_hat", "veces_mediana"]])


print(f"SKU: {len(SKUS)} | Líneas: {LINEAS} | Semanas: {T[0]}–{T[-1]} "
      f"({capacidad['fecha_lunes'].iloc[0]} a {capacidad['fecha_lunes'].iloc[-1]})")
print("ρ_ℓ:", rho)
print("Jarabe extra (L/caja):", jarabe_extra_L_caja)
print(f"Demanda pronosticada total del horizonte: {sum(d_hat.values()):,.0f} cajas")


⚠ Pronósticos atípicos (> 3 × mediana del SKU):


,sku,h,fecha_lunes,d_hat,veces_mediana
0,NEC-D15,11,2026-12-07,11654.0,5.8


SKU: 9 | Líneas: ['L1', 'L2'] | Semanas: 1–13 (2026-09-28 a 2026-12-21)
ρ_ℓ: {'L1': 0.019217, 'L2': 0.031146}
Jarabe extra (L/caja): {0.5: 0.049123, 1.5: 0.027606, 3.0: 0.034005}
Demanda pronosticada total del horizonte: 167,351 cajas


In [74]:
# =====================================================================
# PARTE A1 — BALANCE DE INVENTARIO POR SKU Y SEMANA (con backlogging y merma)
#   s[i,t-1] + (1-ρ_ℓ(i))·x[i,t] + b[i,t] = d[i,t] + s[i,t] + b[i,t-1]
# =====================================================================


# ---------------------------------------------------------------------
# 1. DATOS
# ---------------------------------------------------------------------
datos = maestro.set_index("sku").join(inv_inicial.set_index("sku"))


linea_de = datos["linea_envasado"].to_dict()                                # ℓ(i)
p_var    = datos["costo_variable_caja_clp"].astype(float).to_dict()         # p^i
h        = datos["costo_inventario_caja_semana_clp"].astype(float).to_dict()  # h^i
pi_atr   = datos["penalidad_atraso_caja_semana_clp"].astype(float).to_dict()  # π^i
s0       = datos["inventario_inicial_cajas"].astype(float).to_dict()        # s^i_0
b0       = {i: 0.0 for i in SKUS}                                           # b^i_0 (supuesto)
d        = d_hat                                                            # d^i_t
rend     = {i: 1 - rho[linea_de[i]] for i in SKUS}                          # 1 - ρ_ℓ(i)


tabla_a1_1 = pd.DataFrame({
    "SKU": SKUS,
    "Línea": [linea_de[i] for i in SKUS],
    "ρ_ℓ": [rho[linea_de[i]] for i in SKUS],
    "1 - ρ_ℓ": [rend[i] for i in SKUS],
    "s_0 (cajas)": [s0[i] for i in SKUS],
    "Σ d_t (cajas)": [sum(d[i, t] for t in T) for i in SKUS],
    "p (CLP/caja)": [p_var[i] for i in SKUS],
    "h (CLP/caja·sem)": [h[i] for i in SKUS],
    "π (CLP/caja·sem)": [pi_atr[i] for i in SKUS],
})
print("Tabla A1.1 — Datos del balance por SKU")
display(tabla_a1_1.round(6))


# ---------------------------------------------------------------------
# 2. MODELO, VARIABLES Y BALANCE
# ---------------------------------------------------------------------
# Las partes A2–A8 agregan restricciones a este mismo `modelo` y
# términos nuevos al diccionario `costos`.
modelo = pulp.LpProblem("BdM_LotSizing", pulp.LpMinimize)


x = pulp.LpVariable.dicts("x", (SKUS, T), lowBound=0)  # producción bruta
s = pulp.LpVariable.dicts("s", (SKUS, T), lowBound=0)  # inventario final
b = pulp.LpVariable.dicts("b", (SKUS, T), lowBound=0)  # backlog final


for i in SKUS:
    for t in T:
        s_prev = s0[i] if t == 1 else s[i][t - 1]
        b_prev = b0[i] if t == 1 else b[i][t - 1]
        modelo += (
            s_prev + rend[i] * x[i][t] + b[i][t] == d[i, t] + s[i][t] + b_prev,
            f"balance_{i}_{t}",
        )


costos = {
    "Producción (p·x)": pulp.lpSum(p_var[i] * x[i][t] for i in SKUS for t in T),
    "Inventario (h·s)": pulp.lpSum(h[i] * s[i][t] for i in SKUS for t in T),
    "Atraso (π·b)":     pulp.lpSum(pi_atr[i] * b[i][t] for i in SKUS for t in T),
}
modelo += pulp.lpSum(costos.values())


print(f"\nRestricciones de balance: {len(modelo.constraints)} "
      f"(= {len(SKUS)} SKU × {len(T)} semanas)")
print(f"Variables: {len(modelo.variables())} (x, s, b)")


# ---------------------------------------------------------------------
# 3. RESOLUCIÓN DE PRUEBA (solo A1, sin capacidad)
# ---------------------------------------------------------------------
# Sin capacidad, el óptimo se puede calcular a mano y sirve para
# comprobar el balance. Cada caja demandada en t se atiende de la forma
# más barata:
#   - producirla en t:            p / (1-ρ)
#   - atrasarla hasta el final:   π · (13 - t + 1)
# Si atrasar es más barato, el modelo no la produce. Esto es el efecto
# de fin de horizonte que corrige A7. El resto se produce lot-for-lot.
modelo.solve(pulp.PULP_CBC_CMD(msg=False))
print("\nEstado:", pulp.LpStatus[modelo.status])
print(f"Costo total: ${pulp.value(modelo.objective):,.0f}")
for nombre, termino in costos.items():
    print(f"  {nombre:<18} ${pulp.value(termino):>15,.0f}")


conviene_producir = {(i, t): pi_atr[i] * (T[-1] - t + 1) > p_var[i] / rend[i]
                     for i in SKUS for t in T}


filas = []
for i in SKUS:
    disponible = s0[i]
    for t in T:
        usa = min(disponible, d[i, t])
        disponible -= usa
        filas.append({
            "sku": i, "linea": linea_de[i], "t": t,
            "d": d[i, t],
            "x": x[i][t].varValue,
            "x_vendible": rend[i] * x[i][t].varValue,
            "merma": (1 - rend[i]) * x[i][t].varValue,
            "s": s[i][t].varValue,
            "b": b[i][t].varValue,
            "x_esperado": (d[i, t] - usa) / rend[i] if conviene_producir[i, t] else 0.0,
        })
balance_a1 = pd.DataFrame(filas)


# Comprobaciones
assert np.allclose(balance_a1["x"], balance_a1["x_esperado"], atol=1e-2), \
    "La producción no coincide con el óptimo calculado a mano."
assert not ((balance_a1["s"] > 1e-6) & (balance_a1["b"] > 1e-6)).any(), \
    "Hay inventario y atraso positivos a la vez."
print("Comprobaciones superadas: x coincide con el óptimo calculado a mano "
      "y nunca s > 0 y b > 0 a la vez.")


# Efecto de fin de horizonte
fin_horizonte = pd.DataFrame({
    "SKU": SKUS,
    "p/(1-ρ) (CLP)": [p_var[i] / rend[i] for i in SKUS],
    "semanas de atraso que cuestan menos": [int(p_var[i] / rend[i] // pi_atr[i]) for i in SKUS],
    "deja de producir desde t =": [min([t for t in T if not conviene_producir[i, t]], default="—")
                                   for i in SKUS],
    "b_13 (cajas)": [b[i][T[-1]].varValue for i in SKUS],
})
print("\nTabla A1.2 — Efecto de fin de horizonte sin restricción terminal")
display(fin_horizonte.round(1))


# ---------------------------------------------------------------------
# 4. RESULTADOS
# ---------------------------------------------------------------------
resumen = balance_a1.groupby(["linea", "sku"]).agg(
    demanda=("d", "sum"), produccion_bruta=("x", "sum"),
    produccion_vendible=("x_vendible", "sum"), merma=("merma", "sum"))
resumen.insert(1, "s_0", [s0[i] for i in resumen.index.get_level_values("sku")])
resumen["costo_merma_CLP"] = [m * p_var[i] for (l, i), m in resumen["merma"].items()]
print("\nTabla A1.3 — Producción y merma del horizonte por SKU, sin capacidad (cajas)")
display(resumen.round(1))


por_linea = resumen.groupby(level="linea").sum()
por_linea["merma_%"] = 100 * por_linea["merma"] / por_linea["produccion_bruta"]
print("Tabla A1.4 — Efecto de la merma por línea")
display(por_linea.round(1))


def por_semana(columna_valor):
    tabla = balance_a1.pivot(index=["linea", "sku"], columns="t", values=columna_valor)
    tabla.columns = [f"t={t}" for t in tabla.columns]
    return tabla.round(1)


print("Tabla A1.5 — Demanda d_t (cajas)")
display(por_semana("d"))
print("Tabla A1.6 — Producción bruta x_t (cajas)")
display(por_semana("x"))
print("Tabla A1.7 — Inventario final s_t (cajas)")
display(por_semana("s"))
print("Tabla A1.8 — Backlog final b_t (cajas)")
display(por_semana("b"))


Tabla A1.1 — Datos del balance por SKU


,SKU,Línea,ρ_ℓ,1 - ρ_ℓ,s_0 (cajas),Σ d_t (cajas),p (CLP/caja),h (CLP/caja·sem),π (CLP/caja·sem)
0,NEC-D05,L1,0.019217,0.980783,1024.0,13875.0,3840.0,95.0,1400.0
1,NEC-D15,L2,0.031146,0.968854,3162.0,36281.0,5760.0,95.0,1400.0
2,NEC-D30,L2,0.031146,0.968854,339.0,6851.0,7680.0,95.0,1400.0
3,CIT-L05,L1,0.019217,0.980783,1217.0,30808.0,2820.0,95.0,1400.0
4,CIT-L15,L2,0.031146,0.968854,1282.0,23257.0,4230.0,95.0,1400.0
5,CIT-L30,L2,0.031146,0.968854,255.0,5625.0,5640.0,95.0,1400.0
6,FUN-B05,L1,0.019217,0.980783,2135.0,29659.0,4860.0,95.0,1400.0
7,FUN-B15,L2,0.031146,0.968854,1261.0,18486.0,7290.0,95.0,1400.0
8,FUN-B30,L2,0.031146,0.968854,155.0,2509.0,9720.0,95.0,1400.0



Restricciones de balance: 117 (= 9 SKU × 13 semanas)
Variables: 351 (x, s, b)

Estado: Optimal
Costo total: $688,800,757
  Producción (p·x)   $    515,188,687
  Inventario (h·s)   $        124,070
  Atraso (π·b)       $    173,488,000
Comprobaciones superadas: x coincide con el óptimo calculado a mano y nunca s > 0 y b > 0 a la vez.

Tabla A1.2 — Efecto de fin de horizonte sin restricción terminal


,SKU,p/(1-ρ) (CLP),semanas de atraso que cuestan menos,deja de producir desde t =,b_13 (cajas)
0,NEC-D05,3915.2,2,12,1846.0
1,NEC-D15,5945.2,4,10,18024.0
2,NEC-D30,7926.9,5,9,2635.0
3,CIT-L05,2875.3,2,12,4867.0
4,CIT-L15,4366.0,3,11,5367.0
5,CIT-L30,5821.3,4,10,1962.0
6,FUN-B05,4955.2,3,11,6936.0
7,FUN-B15,7524.4,5,9,7110.0
8,FUN-B30,10032.5,7,7,1351.0



Tabla A1.3 — Producción y merma del horizonte por SKU, sin capacidad (cajas)


demanda     s_0  produccion_bruta  produccion_vendible  merma  \
linea sku                                                                      
L1    CIT-L05  30808.0  1217.0           25208.4              24724.0  484.4   
      FUN-B05  29659.0  2135.0           20991.4              20588.0  403.4   
      NEC-D05  13875.0  1024.0           11220.6              11005.0  215.6   
L2    CIT-L15  23257.0  1282.0           17141.9              16608.0  533.9   
      CIT-L30   5625.0   255.0            3517.6               3408.0  109.6   
      FUN-B15  18486.0  1261.0           10440.2              10115.0  325.2   
      FUN-B30   2509.0   155.0            1035.2               1003.0   32.2   
      NEC-D15  36281.0  3162.0           15580.3              15095.0  485.3   
      NEC-D30   6851.0   339.0            4001.6               3877.0  124.6   

               costo_merma_CLP  
linea sku                       
L1    CIT-L05        1366093.7  
      FUN-B05        1960483.0  
      NEC-D05         828006.9  
L2    CIT-L15        2258404.0  
      CIT-L30         617906.3  
      FUN-B15        2370485.8  
      FUN-B30         313408.8  
      NEC-D15        2795114.1  
      NEC-D30         957196.2

... (6 outputs left out)

Tabla A1.7 — Inventario final s_t (cajas)


t=1  t=2  t=3  t=4  t=5  t=6  t=7  t=8  t=9  t=10  t=11  \
linea sku                                                                   
L1    CIT-L05     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      FUN-B05     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      NEC-D05     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   
L2    CIT-L15     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      CIT-L30     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      FUN-B15     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      FUN-B30     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      NEC-D15  1306.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      NEC-D30     0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   0.0   0.0   

               t=12  t=13  
linea sku                  
L1    CIT-L05   0.0   0.0  
      FUN-B05   0.0   0.0  
      NEC-D05   0.0   0.0  
L2    CIT-L15   0.0   0.0  
      CIT-L30   0.0   0.0  
      FUN-B15   0.0   0.0  
      FUN-B30   0.0   0.0  
      NEC-D15   0.0   0.0  
      NEC-D30   0.0   0.0

Tabla A1.8 — Backlog final b_t (cajas)


t=1  t=2  t=3  t=4  t=5  t=6    t=7    t=8     t=9    t=10  \
linea sku                                                                   
L1    CIT-L05  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0     0.0   
      FUN-B05  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0     0.0   
      NEC-D05  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0     0.0   
L2    CIT-L15  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0     0.0   
      CIT-L30  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0   477.0   
      FUN-B15  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0  1422.0  2844.0   
      FUN-B30  0.0  0.0  0.0  0.0  0.0  0.0  193.0  386.0   579.0   772.0   
      NEC-D15  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0  2025.0   
      NEC-D30  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0   527.0  1054.0   

                  t=11     t=12     t=13  
linea sku                                 
L1    CIT-L05      0.0   2382.0   4867.0  
      FUN-B05   2310.0   4622.0   6936.0  
      NEC-D05      0.0    930.0   1846.0  
L2    CIT-L15   1789.0   3578.0   5367.0  
      CIT-L30    964.0   1459.0   1962.0  
      FUN-B15   4266.0   5688.0   7110.0  
      FUN-B30    965.0   1158.0   1351.0  
      NEC-D15  13679.0  16064.0  18024.0  
      NEC-D30   1581.0   2108.0   2635.0

---
##Parte A2. Variables binarias de preparación y su enlace xit ≤ Mit yi

In [75]:
# =====================================================================
# PARTE A2 — VARIABLES BINARIAS DE PREPARACIÓN Y ENLACE x ≤ M·y
# =====================================================================


# ---------------------------------------------------------------------
# 1. DATOS DE PREPARACIÓN Y CAPACIDAD
# ---------------------------------------------------------------------
q_prep      = {i: 38_000.0 for i in SKUS}                  # q^i: costo cambio de formato (CLP)
beta_linea  = {"L1": 0.4, "L2": 0.6}                       # β_ℓ: horas por preparación
beta        = {i: beta_linea[linea_de[i]] for i in SKUS}
tasa        = datos["tasa_envasado_cajas_h"].astype(float).to_dict()   # r^i (cajas/h)
lote_min    = datos["lote_minimo_cajas"].astype(float).to_dict()
litros_caja = {i: datos.loc[i, "litros_por_caja"]
                  + jarabe_extra_L_caja[datos.loc[i, "formato_L"]] for i in SKUS}   # v^i


cap = capacidad.set_index("t")
H_max_linea = {(l, t): cap.loc[t, f"horas_{l}_regular"] + cap.loc[t, f"horas_{l}_extra_max"]
               for l in LINEAS for t in T}
litros_mezcla_max = ((cap["horas_mezcla_regular"] + cap["horas_mezcla_extra_max"])
                     * cap["tasa_mezcla_L_h"]).max()


# ---------------------------------------------------------------------
# 2. COTA M (mínimo de tres cotas válidas)
# ---------------------------------------------------------------------
def calcular_M(S_fin=None):
    """Devuelve M[i,t] y la tabla de cotas. S_fin = inventario final exigido (A4/A7)."""
    S_fin = S_fin or {i: 0.0 for i in SKUS}
    M_env = {(i, t): tasa[i] * (H_max_linea[linea_de[i], t] - beta[i]) for i in SKUS for t in T}
    M_mez = {i: litros_mezcla_max / litros_caja[i] for i in SKUS}
    M_dem = {i: max(lote_min[i],
                    (sum(d[i, t] for t in T) - s0[i] + S_fin[i]) / rend[i]) for i in SKUS}
    M = {(i, t): min(M_env[i, t], M_mez[i], M_dem[i]) for i in SKUS for t in T}


    def cota_que_manda(i):
        candidatos = {"envasado": min(M_env[i, t] for t in T), "mezcla": M_mez[i], "demanda": M_dem[i]}
        return min(candidatos, key=candidatos.get)


    tabla = pd.DataFrame({
        "SKU": SKUS,
        "Línea": [linea_de[i] for i in SKUS],
        "r (cajas/h)": [tasa[i] for i in SKUS],
        "β (h)": [beta[i] for i in SKUS],
        "M_env mín": [min(M_env[i, t] for t in T) for i in SKUS],
        "M_env máx": [max(M_env[i, t] for t in T) for i in SKUS],
        "M_mez": [M_mez[i] for i in SKUS],
        "M_dem": [M_dem[i] for i in SKUS],
        "M mín": [min(M[i, t] for t in T) for i in SKUS],
        "M máx": [max(M[i, t] for t in T) for i in SKUS],
        "Cota que manda": [cota_que_manda(i) for i in SKUS],
    })
    return M, tabla


M, tabla_a2_1 = calcular_M()
print("Tabla A2.1 — Cotas de M por SKU (cajas)")
display(tabla_a2_1.round({c: (1 if c == "β (h)" else 0) for c in tabla_a2_1.columns[2:-1]}))


tabla_a2_2 = pd.DataFrame([[M[i, t] for t in T] for i in SKUS],
                          index=SKUS, columns=[f"t={t}" for t in T])
print("Tabla A2.2 — M_it final por SKU y semana (cajas)")
display(tabla_a2_2.round(0))


# ---------------------------------------------------------------------
# 3. VARIABLES BINARIAS, ENLACE Y COSTO DE PREPARACIÓN
# ---------------------------------------------------------------------
y = pulp.LpVariable.dicts("y", (SKUS, T), cat="Binary")


nombre_enlace = {(i, t): f"enlace_{i.replace('-', '_')}_{t}" for i in SKUS for t in T}
for i in SKUS:
    for t in T:
        modelo += x[i][t] <= M[i, t] * y[i][t], nombre_enlace[i, t]


costos["Preparación (q·y)"] = pulp.lpSum(q_prep[i] * y[i][t] for i in SKUS for t in T)
modelo.setObjective(pulp.lpSum(costos.values()))


def fijar_M(nuevo_M):
    """Cambia el coeficiente de y en cada enlace (x - M·y ≤ 0)."""
    for i in SKUS:
        for t in T:
            modelo.constraints[nombre_enlace[i, t]][y[i][t]] = -nuevo_M[i, t]


def relajar_y(relajar):
    """Relajación lineal: y continua en [0, 1]; relajar=False la devuelve a binaria."""
    for i in SKUS:
        for t in T:
            y[i][t].cat = pulp.LpContinuous if relajar else pulp.LpInteger


print(f"\nVariables binarias: {len(SKUS) * len(T)} | Restricciones de enlace: {len(SKUS) * len(T)}")


# ---------------------------------------------------------------------
# 4. JUSTIFICACIÓN DE M: M AJUSTADA vs. M ARBITRARIA (10^6)
# ---------------------------------------------------------------------
# Las dos versiones tienen el mismo óptimo entero; lo que cambia es la
# cota de la relajación lineal. Brecha = (z_MIP - z_LP) / z_MIP.
import time
M_grande = {(i, t): 1e6 for i in SKUS for t in T}


def resolver(etiqueta, valores_M, relajado):
    fijar_M(valores_M)
    relajar_y(relajado)
    inicio = time.time()
    modelo.solve(pulp.PULP_CBC_CMD(msg=False))
    return {"Caso": etiqueta, "Estado": pulp.LpStatus[modelo.status],
            "Objetivo (CLP)": pulp.value(modelo.objective),
            "Costo preparación (CLP)": pulp.value(costos["Preparación (q·y)"]),
            "Tiempo (s)": time.time() - inicio}


corridas = [
    resolver("LP — M = 10^6", M_grande, relajado=True),
    resolver("LP — M ajustada", M, relajado=True),
    resolver("MIP — M = 10^6", M_grande, relajado=False),
    resolver("MIP — M ajustada", M, relajado=False),   # queda como solución vigente
]
tabla_a2_3 = pd.DataFrame(corridas).set_index("Caso")
z_mip      = tabla_a2_3.loc["MIP — M ajustada", "Objetivo (CLP)"]
z_lp_big   = tabla_a2_3.loc["LP — M = 10^6", "Objetivo (CLP)"]
tabla_a2_3["Brecha LP vs. MIP"] = (z_mip - tabla_a2_3["Objetivo (CLP)"]) / z_mip
tabla_a2_3["Brecha cerrada vs. M = 10^6"] = (tabla_a2_3["Objetivo (CLP)"] - z_lp_big) / (z_mip - z_lp_big)
tabla_a2_3.loc[tabla_a2_3.index.str.startswith("MIP"), ["Brecha LP vs. MIP", "Brecha cerrada vs. M = 10^6"]] = np.nan
print("\nTabla A2.3 — Relajación lineal y óptimo entero según la cota M")
display(tabla_a2_3.style.format({
    "Objetivo (CLP)": "{:,.0f}", "Costo preparación (CLP)": "{:,.0f}", "Tiempo (s)": "{:.2f}",
    "Brecha LP vs. MIP": "{:.3%}", "Brecha cerrada vs. M = 10^6": "{:.1%}"}, na_rep="—"))
acel = tabla_a2_3.loc["MIP — M = 10^6", "Tiempo (s)"] / tabla_a2_3.loc["MIP — M ajustada", "Tiempo (s)"]
print(f"La M ajustada cierra {tabla_a2_3.loc['LP — M ajustada', 'Brecha cerrada vs. M = 10^6']:.1%} "
      f"de la brecha de la relajación y CBC resuelve el MIP {acel:.0f} veces más rápido.")


# ---------------------------------------------------------------------
# 5. COMPROBACIONES SOBRE LA SOLUCIÓN ENTERA (M ajustada)
# ---------------------------------------------------------------------
assert pulp.LpStatus[modelo.status] == "Optimal"
x_val = {(i, t): x[i][t].varValue for i in SKUS for t in T}
y_val = {(i, t): round(y[i][t].varValue) for i in SKUS for t in T}


assert all(x_val[i, t] <= M[i, t] * y_val[i, t] + 1e-6 for i in SKUS for t in T), \
    "Se viola el enlace x ≤ M·y."
assert all(x_val[i, t] > 1e-6 for i in SKUS for t in T if y_val[i, t] == 1), \
    "Hay preparaciones sin producción."
mip_grande = tabla_a2_3.loc["MIP — M = 10^6", "Objetivo (CLP)"]
assert abs(mip_grande - z_mip) < 1, "M ajustada cambió el óptimo entero: la cota no sería válida."
print("Comprobaciones superadas: x ≤ M·y, toda preparación produce, "
      "y M ajustada da el mismo óptimo entero que M = 10^6.")


print(f"\nCosto total (A1 + A2): ${pulp.value(modelo.objective):,.0f}")
for nombre, termino in costos.items():
    print(f"  {nombre:<20} ${pulp.value(termino):>15,.0f}")


# ---------------------------------------------------------------------
# 6. RESULTADOS: PREPARACIONES Y LOTES
# ---------------------------------------------------------------------
tabla_a2_4 = pd.DataFrame({
    "SKU": SKUS,
    "Línea": [linea_de[i] for i in SKUS],
    "Preparaciones": [sum(y_val[i, t] for t in T) for i in SKUS],
    "Producción bruta (cajas)": [sum(x_val[i, t] for t in T) for i in SKUS],
})
tabla_a2_4["Lote medio (cajas)"] = (tabla_a2_4["Producción bruta (cajas)"]
                                    / tabla_a2_4["Preparaciones"].replace(0, np.nan))
tabla_a2_4["Uso máx. de M (x/M)"] = [max(x_val[i, t] / M[i, t] for t in T) for i in SKUS]
tabla_a2_4["Costo preparación (CLP)"] = tabla_a2_4["Preparaciones"] * 38_000
print("\nTabla A2.4 — Preparaciones y tamaño de lote por SKU")
display(tabla_a2_4.round(2))


def matriz(valores, decimales=0):
    tabla = pd.DataFrame([[valores[i, t] for t in T] for i in SKUS],
                         index=pd.MultiIndex.from_tuples([(linea_de[i], i) for i in SKUS],
                                                         names=["linea", "sku"]),
                         columns=[f"t={t}" for t in T]).sort_index()
    return tabla.round(decimales)


print("Tabla A2.5 — Preparaciones y_it (1 = se prepara la línea para el SKU)")
display(matriz(y_val))
print("Tabla A2.6 — Producción bruta x_it (cajas)")
display(matriz(x_val, 1))
print("Tabla A2.7 — Inventario final s_it (cajas)")
display(matriz({(i, t): s[i][t].varValue for i in SKUS for t in T}, 1))
print("Tabla A2.8 — Backlog final b_it (cajas)")
display(matriz({(i, t): b[i][t].varValue for i in SKUS for t in T}, 1))


Tabla A2.1 — Cotas de M por SKU (cajas)


,SKU,Línea,r (cajas/h),β (h),M_env mín,M_env máx,M_mez,M_dem,M mín,M máx,Cota que manda
0,NEC-D05,L1,180.0,0.4,7848.0,8568.0,24714.0,13103.0,7848.0,8568.0,envasado
1,NEC-D15,L2,210.0,0.6,11634.0,13314.0,16560.0,34184.0,11634.0,13314.0,envasado
2,NEC-D30,L2,130.0,0.6,7202.0,8242.0,12423.0,6721.0,6721.0,6721.0,demanda
3,CIT-L05,L1,180.0,0.4,7848.0,8568.0,24714.0,30171.0,7848.0,8568.0,envasado
4,CIT-L15,L2,210.0,0.6,11634.0,13314.0,16560.0,22681.0,11634.0,13314.0,envasado
5,CIT-L30,L2,130.0,0.6,7202.0,8242.0,12423.0,5543.0,5543.0,5543.0,demanda
6,FUN-B05,L1,180.0,0.4,7848.0,8568.0,24714.0,28063.0,7848.0,8568.0,envasado
7,FUN-B15,L2,210.0,0.6,11634.0,13314.0,16560.0,17779.0,11634.0,13314.0,envasado
8,FUN-B30,L2,130.0,0.6,7202.0,8242.0,12423.0,2430.0,2430.0,2430.0,demanda


Tabla A2.2 — M_it final por SKU y semana (cajas)


,t=1,t=2,t=3,t=4,t=5,t=6,t=7,t=8,t=9,t=10,t=11,t=12,t=13
NEC-D05,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,7848.0
NEC-D15,13314.0,13314.0,13314.0,11634.0,13314.0,13314.0,13314.0,13314.0,11634.0,13314.0,13314.0,13314.0,11634.0
NEC-D30,6721.0,6721.0,6721.0,6721.0,6721.0,6721.0,6721.0,6721.0,6721.0,6721.0,6721.0,6721.0,6721.0
CIT-L05,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,7848.0
CIT-L15,13314.0,13314.0,13314.0,11634.0,13314.0,13314.0,13314.0,13314.0,11634.0,13314.0,13314.0,13314.0,11634.0
CIT-L30,5543.0,5543.0,5543.0,5543.0,5543.0,5543.0,5543.0,5543.0,5543.0,5543.0,5543.0,5543.0,5543.0
FUN-B05,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,8568.0,7848.0
FUN-B15,13314.0,13314.0,13314.0,11634.0,13314.0,13314.0,13314.0,13314.0,11634.0,13314.0,13314.0,13314.0,11634.0
FUN-B30,2430.0,2430.0,2430.0,2430.0,2430.0,2430.0,2430.0,2430.0,2430.0,2430.0,2430.0,2430.0,2430.0



Variables binarias: 117 | Restricciones de enlace: 117

Tabla A2.3 — Relajación lineal y óptimo entero según la cota M


,Estado,Objetivo (CLP),Costo preparación (CLP),Tiempo (s),Brecha LP vs. MIP,Brecha cerrada vs. M = 10^6
Caso,,,,,,
LP — M = 10^6,Optimal,"688,804,904","4,147",0.03,0.435%,0.0%
LP — M ajustada,Optimal,"689,245,514","444,757",0.05,0.371%,14.6%
MIP — M = 10^6,Optimal,"691,814,917","2,888,000",6.60,—,—
MIP — M ajustada,Optimal,"691,814,917","2,888,000",0.13,—,—


... (6 outputs left out)

Tabla A2.7 — Inventario final s_it (cajas)


t=1  t=2    t=3  t=4    t=5  t=6  t=7  t=8  t=9  t=10  t=11  \
linea sku                                                                       
L1    CIT-L05     0.0  0.0    0.0  0.0    0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      FUN-B05     0.0  0.0    0.0  0.0    0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      NEC-D05     0.0  0.0    0.0  0.0    0.0  0.0  0.0  0.0  0.0   0.0   0.0   
L2    CIT-L15     0.0  0.0    0.0  0.0    0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      CIT-L30   354.0  0.0  395.0  0.0    0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      FUN-B15     0.0  0.0    0.0  0.0    0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      FUN-B30   193.0  0.0  193.0  0.0  193.0  0.0  0.0  0.0  0.0   0.0   0.0   
      NEC-D15  1306.0  0.0    0.0  0.0    0.0  0.0  0.0  0.0  0.0   0.0   0.0   
      NEC-D30     0.0  0.0    0.0  0.0    0.0  0.0  0.0  0.0  0.0   0.0   0.0   

               t=12  t=13  
linea sku                  
L1    CIT-L05   0.0   0.0  
      FUN-B05   0.0   0.0  
      NEC-D05   0.0   0.0  
L2    CIT-L15   0.0   0.0  
      CIT-L30   0.0   0.0  
      FUN-B15   0.0   0.0  
      FUN-B30   0.0   0.0  
      NEC-D15   0.0   0.0  
      NEC-D30   0.0   0.0

Tabla A2.8 — Backlog final b_it (cajas)


t=1  t=2  t=3  t=4  t=5  t=6    t=7    t=8     t=9    t=10  \
linea sku                                                                   
L1    CIT-L05  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0     0.0   
      FUN-B05  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0     0.0   
      NEC-D05  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0     0.0   
L2    CIT-L15  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0     0.0   
      CIT-L30  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0   477.0   
      FUN-B15  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0  1422.0  2844.0   
      FUN-B30  0.0  0.0  0.0  0.0  0.0  0.0  193.0  386.0   579.0   772.0   
      NEC-D15  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0     0.0  2025.0   
      NEC-D30  0.0  0.0  0.0  0.0  0.0  0.0    0.0    0.0   527.0  1054.0   

                  t=11     t=12     t=13  
linea sku                                 
L1    CIT-L05      0.0   2382.0   4867.0  
      FUN-B05   2310.0   4622.0   6936.0  
      NEC-D05      0.0    930.0   1846.0  
L2    CIT-L15   1789.0   3578.0   5367.0  
      CIT-L30    964.0   1459.0   1962.0  
      FUN-B15   4266.0   5688.0   7110.0  
      FUN-B30    965.0   1158.0   1351.0  
      NEC-D15  13679.0  16064.0  18024.0  
      NEC-D30   1581.0   2108.0   2635.0



##Parte A3.  Capacidad de envasado

In [80]:
# =====================================================================
# PARTE A3 — CAPACIDAD DE ENVASADO POR LÍNEA Y MEZCLA (CON CIP)
# Ejecutar inmediatamente después de A.2 sin modificar A.1 ni A.2.
# =====================================================================

# 0. Limpieza preventiva para permitir re-ejecución segura
for nom_c in list(modelo.constraints):
    if nom_c.startswith((
        "cap_envasado_",
        "activa_fam_lb_",
        "activa_fam_ub_",
        "cambios_fam_mezcla_",
        "cap_mezcla_",
    )):
        del modelo.constraints[nom_c]

# 1. Parámetros de capacidad y familias
if cap.index.name != "semana" and "semana" in cap.columns:
    cap = cap.set_index("semana")

LINEAS = ["L1", "L2"]
skus_linea = {l: [i for i in SKUS if linea_de[i] == l] for l in LINEAS}

col_fam = "familia" if "familia" in datos.columns else "familia_sabor"
familia_de = (
    datos[col_fam].to_dict()
    if col_fam in datos.columns
    else {i: i.split("-")[0] for i in SKUS}
)
FAMILIAS = sorted(list(set(familia_de.values())))
skus_fam = {f: [i for i in SKUS if familia_de[i] == f] for f in FAMILIAS}

H_reg_linea = {
    (l, t): float(cap.loc[t, f"horas_{l}_regular"])
    for l in LINEAS for t in T
}
tasa_mezcla = {t: float(cap.loc[t, "tasa_mezcla_L_h"]) for t in T}
L_reg_mezcla = {
    t: float(cap.loc[t, "horas_mezcla_regular"]) * tasa_mezcla[t]
    for t in T
}

horas_cip_mezcla = 1.2    # 1,2 h de CIP por familia activa en mezcla
costo_hora_cip   = 74000.0  # CLP 74.000 / h de CIP en mezcla

# Helper de resolución robusto
def resolver_mip(prob, msg=False):
    if hasattr(pulp, "HiGHS") and pulp.HiGHS(msg=msg).available():
        return prob.solve(pulp.HiGHS(msg=msg))
    return prob.solve(pulp.PULP_CBC_CMD(msg=msg, gapRel=0))

# 2. Variables de familia activa (w_ft) y limpiezas CIP (z_t)
if "w" not in globals():
    w = pulp.LpVariable.dicts("w", (FAMILIAS, T), cat="Binary")
if "z" not in globals():
    z = pulp.LpVariable.dicts("z_cambios_fam", T, lowBound=0, cat="Continuous")

# 2.1 Capacidad regular de envasado por línea y semana (horas)
for l in LINEAS:
    for t in T:
        modelo += (
            pulp.lpSum(
                (1.0 / tasa[i]) * x[i][t] + beta[i] * y[i][t]
                for i in skus_linea[l]
            )
            <= H_reg_linea[l, t],
            f"cap_envasado_{l}_{t}",
        )

# 2.2 Activación exacta de familia f en semana t
for f in FAMILIAS:
    for i in skus_fam[f]:
        for t in T:
            modelo += (
                y[i][t] <= w[f][t],
                f"activa_fam_lb_{f}_{i.replace('-', '_')}_{t}",
            )
    for t in T:
        modelo += (
            w[f][t] <= pulp.lpSum(y[i][t] for i in skus_fam[f]),
            f"activa_fam_ub_{f}_{t}",
        )

# 2.3 Limpiezas CIP en mezcla: 1,2 h por cada familia activa en la semana
for t in T:
    modelo += (
        z[t] == pulp.lpSum(w[f][t] for f in FAMILIAS),
        f"cambios_fam_mezcla_{t}",
    )

# 2.4 Capacidad regular de mezcla en litros (jarabe + sobrellenado + CIP)
for t in T:
    modelo += (
        pulp.lpSum(litros_caja[i] * x[i][t] for i in SKUS)
        + (horas_cip_mezcla * tasa_mezcla[t]) * z[t]
        <= L_reg_mezcla[t],
        f"cap_mezcla_{t}",
    )

# 3. Función objetivo con CIP de mezcla (y apertura de los 7 ítems de costo)
costos["Horas extra envasado"] = pulp.lpSum([])
costos["Horas extra mezcla"]   = pulp.lpSum([])
costos["CIP mezcla"] = pulp.lpSum(
    costo_hora_cip * horas_cip_mezcla * z[t] for t in T
)
modelo.setObjective(pulp.lpSum(costos.values()))

# 4. Resolución y reporte estructurado
estado_a3 = resolver_mip(modelo, msg=False)
print("\nEstado (A1 + A2 + A3):", pulp.LpStatus[estado_a3])
print(f"Costo total: ${pulp.value(modelo.objective):,.0f}".replace(",", "."))
for nombre, termino in costos.items():
    val = pulp.value(termino) or 0.0
    print(f"  {nombre:<22} ${val:>15,.0f}".replace(",", "."))

uso_recursos = pd.DataFrame({
    "t": T,
    "Uso_L1_%": [
        100.0 * sum(
            (1.0 / tasa[i]) * (x[i][t].varValue or 0.0)
            + beta[i] * round(y[i][t].varValue or 0.0)
            for i in skus_linea["L1"]
        ) / H_reg_linea["L1", t]
        for t in T
    ],
    "Uso_L2_%": [
        100.0 * sum(
            (1.0 / tasa[i]) * (x[i][t].varValue or 0.0)
            + beta[i] * round(y[i][t].varValue or 0.0)
            for i in skus_linea["L2"]
        ) / H_reg_linea["L2", t]
        for t in T
    ],
    "Uso_Mezcla_%": [
        100.0 * (
            sum(litros_caja[i] * (x[i][t].varValue or 0.0) for i in SKUS)
            + (horas_cip_mezcla * tasa_mezcla[t]) * (z[t].varValue or 0.0)
        ) / L_reg_mezcla[t]
        for t in T
    ],
    "Familias_activas": [
        sum(round(w[f][t].varValue or 0.0) for f in FAMILIAS) for t in T
    ],
}).set_index("t")

print("\nTabla A3.1 — Utilización de capacidad semanal (%)")
display(uso_recursos.round(1))


Estado (A1 + A2 + A3): Optimal
Costo total: $729.485.188
  Producción (p·x)       $    548.716.072
  Inventario (h·s)       $      6.514.734
  Atraso (π·b)           $    168.651.583
  Preparación (q·y)      $      2.850.000
  Horas extra envasado   $              0
  Horas extra mezcla     $              0
  CIP mezcla             $      2.752.800

Tabla A3.1 — Utilización de capacidad semanal (%)


,Uso_L1_%,Uso_L2_%,Uso_Mezcla_%,Familias_activas
t,,,,
1,32.0,48.6,44.4,3
2,79.3,66.2,75.6,3
3,86.8,91.4,91.9,3
4,84.8,77.9,76.7,3
5,85.3,80.7,84.7,3
6,79.9,83.1,96.8,3
7,78.6,85.3,87.4,3
8,88.2,77.6,85.2,3
9,100.0,58.8,71.5,3


 ## Parte A4.  Stock de seguridad

In [84]:
# =====================================================================
# PARTE A4 — STOCK DE SEGURIDAD (SS_it) POR NIVEL DE SERVICIO Y SIGMA_ih
# Ejecutar después de A.3.
# =====================================================================
from scipy.stats import norm

# 0. Limpieza preventiva de restricciones previas de A.4
for nom_c in list(modelo.constraints):
    if nom_c.startswith("stock_seguridad_"):
        del modelo.constraints[nom_c]

# 1. Nivel de servicio objetivo (inv_inicial) y desviación de pronóstico (tabla_c4)
df_inv = inv_inicial.copy()
col_sku_inv = [c for c in df_inv.columns if "sku" in str(c).lower()]
if col_sku_inv:
    df_inv = df_inv.set_index(col_sku_inv[0])

col_ns = [c for c in df_inv.columns if "servicio" in str(c).lower()][0]
nivel_servicio = {}
for i in SKUS:
    ns_val = float(df_inv.loc[i, col_ns])
    nivel_servicio[i] = ns_val / 100.0 if ns_val > 1.0 else ns_val

z_servicio = {i: float(norm.ppf(nivel_servicio[i])) for i in SKUS}

# Lectura robusta de sigma_{i,h} desde tabla_c4 (formato largo o ancho)
df_sig = tabla_c4.copy()
cols_lower = {str(c).strip().lower(): c for c in df_sig.columns}
col_sku_sig = next((orig for low, orig in cols_lower.items() if "sku" in low), None)
col_h_sig   = next((orig for low, orig in cols_lower.items() if low in ("h", "semana", "horizonte", "t")), None)

if col_sku_sig is not None and col_h_sig is not None:
    # Formato largo (filas = SKU x h)
    col_val = next(
        (orig for low, orig in cols_lower.items() if "sigma" in low or "rmse" in low or "std" in low),
        [c for c in df_sig.columns if c not in (col_sku_sig, col_h_sig)][0],
    )
    sig_map = df_sig.set_index([col_sku_sig, col_h_sig])[col_val].to_dict()
    sigma_ih = {(i, t): float(sig_map[i, t]) for i in SKUS for t in T}
else:
    # Formato ancho (filas = SKU, columnas = semanas/horizontes)
    if col_sku_sig is not None:
        df_sig = df_sig.set_index(col_sku_sig)
    elif df_sig.index.dtype == "int64" and len(df_sig) == len(SKUS):
        df_sig.index = SKUS

    # Si las columnas no son 1..13 directamente, tomamos las últimas 13 columnas numéricas
    cols_t_map = {}
    for t in T:
        for cand in (t, str(t), f"h={t}", f"t={t}", f"h_{t}", f"t_{t}", f"S{t}", f"sem_{t}"):
            if cand in df_sig.columns:
                cols_t_map[t] = cand
                break
    if len(cols_t_map) < len(T):
        cols_num = [c for c in df_sig.columns if pd.api.types.is_numeric_dtype(df_sig[c])][-len(T):]
        cols_t_map = dict(zip(T, cols_num))

    sigma_ih = {(i, t): float(df_sig.loc[i, cols_t_map[t]]) for i in SKUS for t in T}

# SS_{i,t} = z_alpha * sigma_{i,h} (lead time de reposición L = 1 semana -> sqrt(1) = 1)
SS = {(i, t): z_servicio[i] * sigma_ih[i, t] for i in SKUS for t in T}

# 2. Restricción de stock de seguridad en el modelo
for i in SKUS:
    for t in T:
        modelo += (
            s[i][t] >= SS[i, t],
            f"stock_seguridad_{i.replace('-', '_')}_{t}",
        )

# 3. Resolución y reporte estructurado
estado_a4 = resolver_mip(modelo, msg=False)
print("\nEstado (A1 + A2 + A3 + A4):", pulp.LpStatus[estado_a4])
print(f"Costo total: ${pulp.value(modelo.objective):,.0f}".replace(",", "."))
for nombre, termino in costos.items():
    val = pulp.value(termino) or 0.0
    print(f"  {nombre:<22} ${val:>15,.0f}".replace(",", "."))

tabla_ss = pd.DataFrame(
    {t: [round(SS[i, t], 1) for i in SKUS] for t in T},
    index=SKUS,
)
tabla_ss.insert(0, "NS_obj", [f"{nivel_servicio[i]*100:.0f}%" for i in SKUS])
tabla_ss.insert(1, "z_alpha", [round(z_servicio[i], 3) for i in SKUS])
print("\nTabla A4.1 — Stock de seguridad requerido SS_it (cajas)")
display(tabla_ss)


Estado (A1 + A2 + A3 + A4): Optimal
Costo total: $729.218.788
  Producción (p·x)       $    548.716.072
  Inventario (h·s)       $      6.514.734
  Atraso (π·b)           $    168.651.583
  Preparación (q·y)      $      2.850.000
  Horas extra envasado   $              0
  Horas extra mezcla     $              0
  CIP mezcla             $      2.486.400

Tabla A4.1 — Stock de seguridad requerido SS_it (cajas)


,NS_obj,z_alpha,1,2,3,4,5,6,7,8,9,10,11,12,13
NEC-D05,95%,1.645,405.8,350.8,486.2,560.0,459.3,407.1,406.4,364.5,504.3,510.4,560.1,330.3,373.4
NEC-D15,95%,1.645,1049.8,1022.2,777.2,980.4,902.4,1171.6,2282.5,2343.6,2261.7,2259.0,1356.9,1199.1,1091.2
NEC-D30,95%,1.645,103.4,158.2,278.1,309.0,299.4,347.5,353.9,339.0,256.8,228.5,242.6,130.4,135.8
CIT-L05,95%,1.645,403.4,562.3,621.0,773.0,762.4,793.9,690.1,661.4,699.0,701.4,730.8,782.0,813.0
CIT-L15,95%,1.645,1036.3,1041.6,1235.1,365.7,404.9,400.3,453.6,595.0,499.3,502.5,743.2,991.1,1060.9
CIT-L30,95%,1.645,112.8,107.7,117.1,77.0,75.6,66.7,72.3,74.5,82.9,85.9,108.9,142.6,150.8
FUN-B05,92%,1.405,494.0,425.4,874.6,789.9,834.8,718.9,815.6,727.3,717.8,628.9,649.0,540.5,484.4
FUN-B15,92%,1.405,416.7,422.9,482.3,600.3,624.7,592.8,549.7,561.1,552.4,584.8,297.8,405.2,410.2
FUN-B30,92%,1.405,276.4,185.8,305.2,255.4,240.4,307.5,293.9,352.6,284.0,342.8,287.9,333.0,251.9


## Parte A5. Horas extra

In [85]:
# ============================================================
# A.5 — HORAS EXTRA: DECISIÓN, LÍMITES Y COSTOS
# Ejecutar después de A.4.
#
# Datos: capacidad_horizonte.csv
# Supuesto: el costo por hora extra de L1 es igual al de L2.
# Los resultados corresponden al modelo incorporado hasta A.5.
# ============================================================

# 1. Variables de decisión
if "HE_linea" not in globals():
    HE_linea = pulp.LpVariable.dicts("HE_linea", (LINEAS, T), lowBound=0)

if "HE_mezcla" not in globals():
    HE_mezcla = pulp.LpVariable.dicts("HE_mezcla", T, lowBound=0)

# 2. Horas extra máximas y capacidad de envasado
for l in LINEAS:
    for t in T:
        HE_linea[l][t].upBound = float(cap.loc[t, f"horas_{l}_extra_max"])
        nombre = f"cap_envasado_{l}_{t}"
        if nombre in modelo.constraints:
            del modelo.constraints[nombre]

        modelo += (
            pulp.lpSum(
                (1.0 / tasa[i]) * x[i][t] + beta[i] * y[i][t]
                for i in skus_linea[l]
            )
            <= H_reg_linea[l, t] + HE_linea[l][t],
            nombre,
        )

# 3. Horas extra máximas y capacidad de mezcla (con CIP de 1,2 h/familia activa)
for t in T:
    HE_mezcla[t].upBound = float(cap.loc[t, "horas_mezcla_extra_max"])

    if f"cambios_fam_mezcla_{t}" in modelo.constraints:
        del modelo.constraints[f"cambios_fam_mezcla_{t}"]
    modelo += (
        z[t] == pulp.lpSum(w[f][t] for f in FAMILIAS),
        f"cambios_fam_mezcla_{t}",
    )

    nombre = f"cap_mezcla_{t}"
    if nombre in modelo.constraints:
        del modelo.constraints[nombre]

    modelo += (
        pulp.lpSum(litros_caja[i] * x[i][t] for i in SKUS)
        + horas_cip_mezcla * tasa_mezcla[t] * z[t]
        <= L_reg_mezcla[t] + tasa_mezcla[t] * HE_mezcla[t],
        nombre,
    )

# 4. Costos de horas extra (tarifa completa según capacidad_horizonte.csv)
costos["Horas extra envasado"] = pulp.lpSum(
    float(cap.loc[t, "costo_hora_extra_L2_clp"]) * HE_linea[l][t]
    for l in LINEAS
    for t in T
)
costos["Horas extra mezcla"] = pulp.lpSum(
    float(cap.loc[t, "costo_hora_extra_mezcla_clp"]) * HE_mezcla[t]
    for t in T
)
modelo.setObjective(pulp.lpSum(costos.values()))

# 5. Resolución del modelo
estado = resolver_mip(modelo, msg=False)

print("A.5 — Horas extra")
print("Estado del modelo:", pulp.LpStatus[estado])

# 6. Tabla semanal de horas extra y desglose completo de costos
if estado == pulp.LpStatusOptimal:
    filas = []
    for t in T:
        horas_L1 = max(0.0, pulp.value(HE_linea["L1"][t]) or 0.0)
        horas_L2 = max(0.0, pulp.value(HE_linea["L2"][t]) or 0.0)
        horas_mezcla = max(0.0, pulp.value(HE_mezcla[t]) or 0.0)

        max_L1 = float(cap.loc[t, "horas_L1_extra_max"])
        max_L2 = float(cap.loc[t, "horas_L2_extra_max"])
        max_mezcla = float(cap.loc[t, "horas_mezcla_extra_max"])

        costo_hora_linea = float(cap.loc[t, "costo_hora_extra_L2_clp"])
        costo_hora_mezcla = float(cap.loc[t, "costo_hora_extra_mezcla_clp"])

        costo_semanal = (
            (horas_L1 + horas_L2) * costo_hora_linea
            + horas_mezcla * costo_hora_mezcla
        )

        assert horas_L1 <= max_L1 + 0.0001
        assert horas_L2 <= max_L2 + 0.0001
        assert horas_mezcla <= max_mezcla + 0.0001

        filas.append({
            "Semana": t,
            "L1 usadas (h)": horas_L1,
            "L1 máx. (h)": max_L1,
            "L2 usadas (h)": horas_L2,
            "L2 máx. (h)": max_L2,
            "Mezcla usadas (h)": horas_mezcla,
            "Mezcla máx. (h)": max_mezcla,
            "Costo HE ($)": costo_semanal,
        })

    tabla_HE = pd.DataFrame(filas)
    total_horas = (
        tabla_HE["L1 usadas (h)"].sum()
        + tabla_HE["L2 usadas (h)"].sum()
        + tabla_HE["Mezcla usadas (h)"].sum()
    )
    total_costo_HE = tabla_HE["Costo HE ($)"].sum()

    fila_total = {"Semana": "Total"}
    for columna in tabla_HE.columns[1:]:
        fila_total[columna] = tabla_HE[columna].sum()
    tabla_HE = pd.concat([tabla_HE, pd.DataFrame([fila_total])], ignore_index=True)

    formatos = {c: "{:.2f}" for c in tabla_HE.columns if "(h)" in c}
    formatos["Costo HE ($)"] = "{:,.0f}"

    print("\nHoras extra utilizadas frente al máximo permitido")
    display(
        tabla_HE.style
        .hide(axis="index")
        .format(formatos, thousands=".", decimal=",")
        .set_properties(**{
            "text-align": "center",
            "padding": "4px 6px",
            "border": "1px solid #888",
        })
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "center"), ("padding", "4px 6px"), ("border", "1px solid #888")]},
            {"selector": "table", "props": [("border-collapse", "collapse")]},
            {"selector": "tbody tr:last-child", "props": [("font-weight", "bold")]},
        ])
    )

    print(f"\nCosto total de horas extra: ${total_costo_HE:,.0f}".replace(",", "."))
    print(f"Costo total del plan hasta A.5: ${pulp.value(modelo.objective):,.0f}".replace(",", "."))
    for nombre, termino in costos.items():
        val = pulp.value(termino) or 0.0
        print(f"  {nombre:<22} ${val:>15,.0f}".replace(",", "."))
else:
    print("\nNo se obtuvo una solución óptima.")

A.5 — Horas extra
Estado del modelo: Optimal

Horas extra utilizadas frente al máximo permitido


Semana,L1 usadas (h),L1 máx. (h),L2 usadas (h),L2 máx. (h),Mezcla usadas (h),Mezcla máx. (h),Costo HE ($)
1,"0,00","8,00","0,00","16,00","0,00","10,00",0
2,"0,00","8,00","0,00","16,00","0,00","10,00",0
3,"0,00","8,00","0,00","16,00","0,00","10,00",0
4,"0,00","8,00","0,00","16,00","0,00","10,00",0
5,"0,00","8,00","0,00","16,00","0,00","10,00",0
6,"0,00","8,00","0,00","16,00","0,00","10,00",0
7,"0,00","8,00","0,00","16,00","0,00","10,00",0
8,"0,00","8,00","0,00","16,00","0,00","10,00",0
9,"0,00","8,00","0,00","16,00","0,00","10,00",0
10,"0,00","8,00","0,00","16,00","0,00","10,00",0



Costo total de horas extra: $0
Costo total del plan hasta A.5: $729.218.788
  Producción (p·x)       $    548.716.072
  Inventario (h·s)       $      6.514.734
  Atraso (π·b)           $    168.651.583
  Preparación (q·y)      $      2.850.000
  Horas extra envasado   $              0
  Horas extra mezcla     $              0
  CIP mezcla             $      2.486.400


## Parte A6.  Lote mínimo

In [83]:
# ============================================================
# A.6 — LOTES MÍNIMOS Y ANTICIPACIÓN DEL JARABE
# Ejecutar después de A.5 (idempotente: seguro de re-ejecutar).
# ============================================================

# 0. Limpieza de restricciones previas de A.6 (si se re-ejecuta)
prefijos_a6 = (
    "lote_minimo_",
    "jarabe_inicial_",
    "activa_fam_lb_",
    "activa_fam_ub_",
    "cambios_fam_mezcla_",
    "cap_mezcla_",
    "jarabe_anticipado_",
    "sin_preparacion_semana14_",
    "familia_mezcla_inferior_",
    "familia_mezcla_superior_",
    "familia_mezcla_final_",
    "mezcla_activa_inferior_",
    "mezcla_activa_superior_",
    "cantidad_cip_",
)
for nombre in list(modelo.constraints):
    if nombre.startswith(prefijos_a6):
        del modelo.constraints[nombre]

# 1. Datos y verificaciones
assert T == list(range(1, 14)), "Se requieren semanas del 1 al 13."
assert (
    datos["lead_time_jarabe_semanas"].astype(float) == 1
).all(), "Este bloque considera un plazo de jarabe de una semana."

lote_min = datos["lote_minimo_cajas"].astype(float).to_dict()

# 2. Lotes mínimos (con nombre de restricción sanitizado)
for i in SKUS:
    sku_limpio = i.replace("-", "_")
    for t in T:
        modelo += (
            x[i][t] >= lote_min[i] * y[i][t],
            f"lote_minimo_{sku_limpio}_{t}",
        )

# 3. Disponibilidad inicial de jarabe — Supuesto de horizonte rodante
# El jarabe envasado en t=1 fue preparado en t=0 (fuera del horizonte),
# sin techo artificial que bloquee adelantar inventario en semana 1.

# 4. Actualizar la cota M con la capacidad física máxima de envasado
H_max_linea = {
    (l, t): float(cap.loc[t, f"horas_{l}_regular"]) + float(cap.loc[t, f"horas_{l}_extra_max"])
    for l in LINEAS for t in T
}
M = {
    (i, t): max(0.0, tasa[i] * (H_max_linea[linea_de[i], t] - beta[i]))
    for i in SKUS
    for t in T
}
fijar_M(M)

# 5. Variables de preparación de jarabe q_jarabe[i][t] (litros en semana t)
if "q_jarabe" not in globals():
    q_jarabe = pulp.LpVariable.dicts("q_jarabe_L", (SKUS, T), lowBound=0)

# 6. Anticipación de una semana: lo preparado en t abastece t+1
for i in SKUS:
    sku_limpio = i.replace("-", "_")
    for t in T:
        if t < T[-1]:
            modelo += (
                q_jarabe[i][t] == litros_caja[i] * x[i][t + 1],
                f"jarabe_anticipado_{sku_limpio}_{t}",
            )
        else:
            modelo += (
                q_jarabe[i][t] == 0,
                f"sin_preparacion_semana14_{sku_limpio}",
            )

# 7. Familias activas en la semana real de mezcla (abastecen t+1)
for f in FAMILIAS:
    for t in T:
        if t < T[-1]:
            for i in skus_fam[f]:
                sku_limpio = i.replace("-", "_")
                modelo += (
                    y[i][t + 1] <= w[f][t],
                    f"familia_mezcla_inferior_{sku_limpio}_{t}",
                )
            modelo += (
                w[f][t] <= pulp.lpSum(y[i][t + 1] for i in skus_fam[f]),
                f"familia_mezcla_superior_{f}_{t}",
            )
        else:
            modelo += (
                w[f][t] == 0,
                f"familia_mezcla_final_{f}",
            )

# 8. Limpiezas CIP (1,2 h por familia activa) y capacidad de mezcla en t
for t in T:
    modelo += (
        z[t] == pulp.lpSum(w[f][t] for f in FAMILIAS),
        f"cantidad_cip_{t}",
    )
    modelo += (
        pulp.lpSum(q_jarabe[i][t] for i in SKUS)
        + horas_cip_mezcla * tasa_mezcla[t] * z[t]
        <= L_reg_mezcla[t] + tasa_mezcla[t] * HE_mezcla[t],
        f"cap_mezcla_{t}",
    )

# 9. Costo de CIP de mezcla ($74.000 / h)
costos["CIP mezcla"] = pulp.lpSum(
    costo_hora_cip * horas_cip_mezcla * z[t] for t in T
)
modelo.setObjective(pulp.lpSum(costos.values()))

# 10. Resolver el modelo hasta A.6
estado = resolver_mip(modelo, msg=False)
print("A.6 — Lotes mínimos y anticipación del jarabe")
print("Estado:", pulp.LpStatus[estado])

if estado != pulp.LpStatusOptimal:
    raise RuntimeError(
        "No se obtuvo una solución óptima. Revisar la factibilidad."
    )

# 11. Verificaciones de cumplimiento
tolerancia = 0.02
assert all(
    x[i][t].value() + tolerancia >= lote_min[i] * y[i][t].value()
    for i in SKUS for t in T
), "Hay un lote que no cumple el mínimo."

assert all(
    abs(q_jarabe[i][t].value() - litros_caja[i] * x[i][t + 1].value()) <= tolerancia
    for i in SKUS for t in T[:-1]
), "Hay una inconsistencia en la anticipación del jarabe."

assert all(
    restriccion.valid(tolerancia)
    for restriccion in modelo.constraints.values()
), "Hay una restricción que no se cumple dentro de la tolerancia."

# 12. Formato de tablas y resumen final
def mostrar_tabla_a6(titulo, tabla, formatos):
    print("\n" + titulo)
    display(
        tabla.style
        .hide(axis="index")
        .format(formatos, thousands=".", decimal=",", na_rep="—")
        .set_properties(**{"text-align": "center", "padding": "4px 6px", "border": "1px solid #888"})
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "center"), ("padding", "4px 6px"), ("border", "1px solid #888")]},
            {"selector": "table", "props": [("border-collapse", "collapse")]},
        ])
    )

tabla_lotes_a6 = pd.DataFrame([
    {
        "SKU": i,
        "Mínimo exigido (cajas)": lote_min[i],
        "Menor lote producido (cajas)": (
            min([x[i][t].value() for t in T if y[i][t].value() > 0.5])
            if any(y[i][t].value() > 0.5 for t in T) else np.nan
        ),
        "Semanas con producción": sum(1 for t in T if y[i][t].value() > 0.5),
        "Cumple": "Sí" if any(y[i][t].value() > 0.5 for t in T) else "Sin producción",
    }
    for i in SKUS
])
mostrar_tabla_a6(
    "Lotes mínimos por SKU",
    tabla_lotes_a6,
    {"Mínimo exigido (cajas)": "{:.0f}", "Menor lote producido (cajas)": "{:.2f}", "Semanas con producción": "{:.0f}"},
)

tabla_inicial_a6 = pd.DataFrame([
    {
        "SKU": i,
        "Cajas envasadas semana 1": x[i][1].value(),
        "Jarabe utilizado en semana 1 (L)": litros_caja[i] * x[i][1].value(),
    }
    for i in SKUS
])
mostrar_tabla_a6(
    "Jarabe utilizado en semana 1 (preparado en t=0)",
    tabla_inicial_a6,
    {"Cajas envasadas semana 1": "{:,.2f}", "Jarabe utilizado en semana 1 (L)": "{:,.2f}"},
)

tabla_jarabe_a6 = pd.DataFrame([
    {
        "Semana de preparación": t,
        "Semana de envasado": t + 1,
        "Preparado (L)": sum(q_jarabe[i][t].value() for i in SKUS),
        "Consumo siguiente semana (L)": sum(litros_caja[i] * x[i][t + 1].value() for i in SKUS),
        "Cumple anticipación": (
            "Sí"
            if abs(
                sum(q_jarabe[i][t].value() for i in SKUS)
                - sum(litros_caja[i] * x[i][t + 1].value() for i in SKUS)
            ) <= tolerancia
            else "No"
        ),
    }
    for t in T[:-1]
])
mostrar_tabla_a6(
    "Preparación y consumo de jarabe",
    tabla_jarabe_a6,
    {
        "Semana de preparación": "{:.0f}",
        "Semana de envasado": "{:.0f}",
        "Preparado (L)": "{:,.2f}",
        "Consumo siguiente semana (L)": "{:,.2f}",
    },
)

costo_HE_actual = (
    (pulp.value(costos["Horas extra envasado"]) or 0.0)
    + (pulp.value(costos["Horas extra mezcla"]) or 0.0)
)
print(f"\nCosto total del plan hasta A.6: ${pulp.value(modelo.objective):,.0f}".replace(",", "."))
print(f"Costo de horas extra actualizado: ${costo_HE_actual:,.0f}".replace(",", "."))
for nombre, termino in costos.items():
    val = pulp.value(termino) or 0.0
    print(f"  {nombre:<22} ${val:>15,.0f}".replace(",", "."))

A.6 — Lotes mínimos y anticipación del jarabe
Estado: Optimal

Lotes mínimos por SKU


SKU,Mínimo exigido (cajas),Menor lote producido (cajas),Semanas con producción,Cumple
NEC-D05,350,"350,00",10,Sí
NEC-D15,600,"815,40",8,Sí
NEC-D30,140,"194,04",8,Sí
CIT-L05,350,"383,37",11,Sí
CIT-L15,430,"523,30",10,Sí
CIT-L30,70,"426,28",7,Sí
FUN-B05,550,"550,00",10,Sí
FUN-B15,360,"360,00",8,Sí
FUN-B30,30,"238,43",3,Sí



Jarabe utilizado en semana 1 (preparado en t=0)


SKU,Cajas envasadas semana 1,Jarabe utilizado en semana 1 (L)
NEC-D05,"350,00","2.117,19"
NEC-D15,"-0,00","-0,00"
NEC-D30,"194,04","2.335,12"
CIT-L05,"383,37","2.319,04"
CIT-L15,"523,30","4.724,13"
CIT-L30,"442,79","5.328,55"
FUN-B05,"550,00","3.327,02"
FUN-B15,"360,00","3.249,94"
FUN-B30,"238,43","2.869,22"



Preparación y consumo de jarabe


Semana de preparación,Semana de envasado,Preparado (L),Consumo siguiente semana (L),Cumple anticipación
1,2,"71.886,68","71.886,68",Sí
2,3,"101.937,98","101.937,98",Sí
3,4,"92.542,19","92.542,19",Sí
4,5,"98.933,58","98.933,58",Sí
5,6,"98.988,91","98.988,91",Sí
6,7,"91.733,85","91.733,85",Sí
7,8,"98.475,41","98.475,41",Sí
8,9,"82.823,49","82.823,49",Sí
9,10,"53.946,83","53.946,83",Sí
10,11,"22.400,89","22.400,89",Sí



Costo total del plan hasta A.6: $694.421.921
Costo de horas extra actualizado: $0
  Producción (p·x)       $    515.188.689
  Inventario (h·s)       $        408.832
  Atraso (π·b)           $    173.488.000
  Preparación (q·y)      $      2.850.000
  Horas extra envasado   $              0
  Horas extra mezcla     $              0
  CIP mezcla             $      2.486.400


## Parte A7.  Restricción de fin de horizonte

## Parte A8. Pedidos en firme